# 04 — Feature Engineering


## Setup

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install -q duckdb


In [5]:
import os
import duckdb
import pandas as pd

BASE_PATH = "/content/drive/MyDrive/Reflow"
PROCESSED_PATH = f"{BASE_PATH}/data/processed"
FEATURES_PATH = f"{BASE_PATH}/data/features"

os.makedirs(FEATURES_PATH, exist_ok=True)
os.makedirs("/content/duckdb_tmp", exist_ok=True)

TRAIN_DAILY_PATH = f"{PROCESSED_PATH}/train_daily_enriched.parquet"
EVAL_DAILY_PATH = f"{PROCESSED_PATH}/eval_daily_enriched.parquet"
TRAIN_HOURLY_PATH = f"{PROCESSED_PATH}/train_hourly.parquet"
EVAL_HOURLY_PATH = f"{PROCESSED_PATH}/eval_hourly.parquet"

TRAIN_FEATURES_PATH = f"{FEATURES_PATH}/train_features.parquet"
EVAL_FEATURES_PATH = f"{FEATURES_PATH}/eval_features.parquet"

con = duckdb.connect()
con.execute("PRAGMA memory_limit='4GB'")
con.execute("PRAGMA temp_directory='/content/duckdb_tmp'")  # spill ke disk LOKAL Colab, bukan ke Drive (jauh lebih cepat)

print("DuckDB version:", duckdb.__version__)


DuckDB version: 1.3.2


##  `expected_sales_baseline`



In [6]:
MIN_SAMPLES = 5

con.execute(f"""
CREATE OR REPLACE TABLE level4 AS
SELECT store_id, product_id, hour, weekday,
       AVG(observed_sales) AS mean_l4, COUNT(*) AS count_l4
FROM read_parquet('{TRAIN_HOURLY_PATH}')
WHERE is_stockout = 0
GROUP BY store_id, product_id, hour, weekday;
""")

con.execute(f"""
CREATE OR REPLACE TABLE level3 AS
SELECT store_id, product_id, hour,
       AVG(observed_sales) AS mean_l3, COUNT(*) AS count_l3
FROM read_parquet('{TRAIN_HOURLY_PATH}')
WHERE is_stockout = 0
GROUP BY store_id, product_id, hour;
""")

con.execute(f"""
CREATE OR REPLACE TABLE level2 AS
SELECT first_category_id, hour, weekday,
       AVG(observed_sales) AS mean_l2, COUNT(*) AS count_l2
FROM read_parquet('{TRAIN_HOURLY_PATH}')
WHERE is_stockout = 0
GROUP BY first_category_id, hour, weekday;
""")

con.execute(f"""
CREATE OR REPLACE TABLE level1 AS
SELECT hour, weekday,
       AVG(observed_sales) AS mean_l1, COUNT(*) AS count_l1
FROM read_parquet('{TRAIN_HOURLY_PATH}')
WHERE is_stockout = 0
GROUP BY hour, weekday;
""")

print("level4 (store-product-hour-weekday) groups:", con.execute("SELECT COUNT(*) FROM level4").fetchone()[0])
print("level3 (store-product-hour) groups        :", con.execute("SELECT COUNT(*) FROM level3").fetchone()[0])
print("level2 (category-hour-weekday) groups      :", con.execute("SELECT COUNT(*) FROM level2").fetchone()[0])
print("level1 (hour-weekday) groups                :", con.execute("SELECT COUNT(*) FROM level1").fetchone()[0])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

level4 (store-product-hour-weekday) groups: 8377281
level3 (store-product-hour) groups        : 1198744
level2 (category-hour-weekday) groups      : 5376
level1 (hour-weekday) groups                : 168


In [7]:
con.execute(f"""
CREATE OR REPLACE TABLE all_keys AS
SELECT DISTINCT store_id, product_id, first_category_id, hour, weekday
FROM (
    SELECT store_id, product_id, first_category_id, hour, weekday FROM read_parquet('{TRAIN_HOURLY_PATH}')
    UNION ALL
    SELECT store_id, product_id, first_category_id, hour, weekday FROM read_parquet('{EVAL_HOURLY_PATH}')
);
""")

print("Jumlah kombinasi unik store-product-hour-weekday:", con.execute("SELECT COUNT(*) FROM all_keys").fetchone()[0])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Jumlah kombinasi unik store-product-hour-weekday: 8400000


In [8]:
con.execute(f"""
CREATE OR REPLACE TABLE baseline_lookup AS
SELECT
    k.store_id, k.product_id, k.hour, k.weekday,
    CAST(
        CASE
            WHEN l4.count_l4 >= {MIN_SAMPLES} THEN l4.mean_l4
            WHEN l3.count_l3 >= {MIN_SAMPLES} THEN l3.mean_l3
            WHEN l2.count_l2 >= {MIN_SAMPLES} THEN l2.mean_l2
            ELSE l1.mean_l1
        END AS FLOAT
    ) AS expected_sales_baseline,
    CASE
        WHEN l4.count_l4 >= {MIN_SAMPLES} THEN 'store_product_hour_weekday'
        WHEN l3.count_l3 >= {MIN_SAMPLES} THEN 'store_product_hour'
        WHEN l2.count_l2 >= {MIN_SAMPLES} THEN 'category_hour_weekday'
        ELSE 'global_hour_weekday'
    END AS baseline_source
FROM all_keys k
LEFT JOIN level4 l4 ON k.store_id=l4.store_id AND k.product_id=l4.product_id AND k.hour=l4.hour AND k.weekday=l4.weekday
LEFT JOIN level3 l3 ON k.store_id=l3.store_id AND k.product_id=l3.product_id AND k.hour=l3.hour
LEFT JOIN level2 l2 ON k.first_category_id=l2.first_category_id AND k.hour=l2.hour AND k.weekday=l2.weekday
LEFT JOIN level1 l1 ON k.hour=l1.hour AND k.weekday=l1.weekday;
""")

print("Distribusi sumber baseline yang dipakai:")
print(con.execute("SELECT baseline_source, COUNT(*) AS n FROM baseline_lookup GROUP BY baseline_source ORDER BY n DESC").df())

con.execute(f"COPY baseline_lookup TO '{FEATURES_PATH}/expected_sales_baseline.parquet' (FORMAT PARQUET);")
print("\nTersimpan:", f"{FEATURES_PATH}/expected_sales_baseline.parquet")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Distribusi sumber baseline yang dipakai:
              baseline_source        n
0  store_product_hour_weekday  8012862
1          store_product_hour   370072
2       category_hour_weekday    17066


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Tersimpan: /content/drive/MyDrive/Reflow/data/features/expected_sales_baseline.parquet


## Fitur Durasi & Severity Stockout



In [9]:
con.execute(f"""
CREATE OR REPLACE TABLE run_features AS
WITH combined AS (
    SELECT store_id, product_id, datetime, is_stockout FROM read_parquet('{TRAIN_HOURLY_PATH}')
    UNION ALL
    SELECT store_id, product_id, datetime, is_stockout FROM read_parquet('{EVAL_HOURLY_PATH}')
),
with_prev AS (
    SELECT *,
        LAG(is_stockout) OVER (PARTITION BY store_id, product_id ORDER BY datetime) AS prev_status
    FROM combined
),
with_block AS (
    SELECT *,
        SUM(CASE WHEN is_stockout != prev_status OR prev_status IS NULL THEN 1 ELSE 0 END)
            OVER (PARTITION BY store_id, product_id ORDER BY datetime
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS block_id
    FROM with_prev
),
with_position AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY store_id, product_id, block_id ORDER BY datetime) AS run_position
    FROM with_block
)
SELECT
    store_id, product_id, datetime,
    CASE WHEN is_stockout = 1 THEN run_position ELSE 0 END AS hours_into_stockout,
    CASE WHEN is_stockout = 0 THEN run_position ELSE 0 END AS hours_since_available,
    CASE WHEN is_stockout = 1 THEN block_id ELSE -1 END AS stockout_episode_id
FROM with_position;
""")

print("Total baris run_features:", con.execute("SELECT COUNT(*) FROM run_features").fetchone()[0])
print(con.execute("""
    SELECT * FROM run_features
    ORDER BY store_id, product_id, datetime
    LIMIT 15
""").df())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total baris run_features: 116400000
    store_id  product_id            datetime  hours_into_stockout  \
0          0           4 2024-03-28 00:00:00                    1   
1          0           4 2024-03-28 01:00:00                    2   
2          0           4 2024-03-28 02:00:00                    3   
3          0           4 2024-03-28 03:00:00                    0   
4          0           4 2024-03-28 04:00:00                    0   
5          0           4 2024-03-28 05:00:00                    0   
6          0           4 2024-03-28 06:00:00                    0   
7          0           4 2024-03-28 07:00:00                    0   
8          0           4 2024-03-28 08:00:00                    0   
9          0           4 2024-03-28 09:00:00                    1   
10         0           4 2024-03-28 10:00:00                    2   
11         0           4 2024-03-28 11:00:00                    3   
12         0           4 2024-03-28 12:00:00                    4  

## Fitur Lag & Rolling (level harian)



In [10]:
con.execute(f"""
CREATE OR REPLACE TABLE daily_lag_features AS
WITH daily_combined AS (
    SELECT store_id, product_id, dt, sale_amount FROM read_parquet('{TRAIN_DAILY_PATH}')
    UNION ALL
    SELECT store_id, product_id, dt, sale_amount FROM read_parquet('{EVAL_DAILY_PATH}')
)
SELECT
    store_id, product_id, dt AS date,
    LAG(sale_amount, 7) OVER w AS sales_lag_7d,
    LAG(sale_amount, 14) OVER w AS sales_lag_14d,
    AVG(sale_amount) OVER (PARTITION BY store_id, product_id ORDER BY dt
                            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS rolling_mean_7d,
    AVG(sale_amount) OVER (PARTITION BY store_id, product_id ORDER BY dt
                            ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING) AS rolling_mean_14d
FROM daily_combined
WINDOW w AS (PARTITION BY store_id, product_id ORDER BY dt);
""")

print("Total baris daily_lag_features:", con.execute("SELECT COUNT(*) FROM daily_lag_features").fetchone()[0])

pct_null_lag7 = con.execute(
    "SELECT AVG(CASE WHEN sales_lag_7d IS NULL THEN 1.0 ELSE 0.0 END) FROM daily_lag_features"
).fetchone()[0]
print(f"% baris sales_lag_7d NULL (wajar utk 7 hari pertama tiap store-product): {pct_null_lag7 * 100:.2f}%")

con.execute(f"COPY daily_lag_features TO '{FEATURES_PATH}/daily_lag_features.parquet' (FORMAT PARQUET);")
print("Tersimpan:", f"{FEATURES_PATH}/daily_lag_features.parquet")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total baris daily_lag_features: 4850000
% baris sales_lag_7d NULL (wajar utk 7 hari pertama tiap store-product): 7.22%
Tersimpan: /content/drive/MyDrive/Reflow/data/features/daily_lag_features.parquet


##Gabungkan Semua Fitur



In [11]:
def assemble(hourly_path, output_path):
    con.execute(f"""
    COPY (
        SELECT
            h.*,
            CAST(h.datetime AS DATE) AS date,
            b.expected_sales_baseline,
            b.baseline_source,
            r.hours_into_stockout,
            r.hours_since_available,
            r.stockout_episode_id,
            d.sales_lag_7d,
            d.sales_lag_14d,
            d.rolling_mean_7d,
            d.rolling_mean_14d
        FROM read_parquet('{hourly_path}') h
        LEFT JOIN baseline_lookup b
            ON h.store_id = b.store_id AND h.product_id = b.product_id
           AND h.hour = b.hour AND h.weekday = b.weekday
        LEFT JOIN run_features r
            ON h.store_id = r.store_id AND h.product_id = r.product_id AND h.datetime = r.datetime
        LEFT JOIN daily_lag_features d
            ON h.store_id = d.store_id AND h.product_id = d.product_id AND CAST(h.datetime AS DATE) = d.date
    ) TO '{output_path}' (FORMAT PARQUET);
    """)
    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{output_path}')").fetchone()[0]
    print(f"Selesai -> {output_path} ({n:,} baris)")


print("=== Assemble TRAIN features ===")
assemble(TRAIN_HOURLY_PATH, TRAIN_FEATURES_PATH)


=== Assemble TRAIN features ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Selesai -> /content/drive/MyDrive/Reflow/data/features/train_features.parquet (108,000,000 baris)


In [12]:
print("=== Assemble EVAL features ===")
assemble(EVAL_HOURLY_PATH, EVAL_FEATURES_PATH)


=== Assemble EVAL features ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Selesai -> /content/drive/MyDrive/Reflow/data/features/eval_features.parquet (8,400,000 baris)


## Validasi akhir

In [13]:
!ls -lh "{TRAIN_FEATURES_PATH}" "{EVAL_FEATURES_PATH}"


-rw------- 1 root root  62M Aug 20 02:05 /content/drive/MyDrive/Reflow/data/features/eval_features.parquet
-rw------- 1 root root 1.5G Aug 20 02:03 /content/drive/MyDrive/Reflow/data/features/train_features.parquet


In [14]:
sample = con.execute(f"""
    SELECT * FROM read_parquet('{TRAIN_FEATURES_PATH}')
    USING SAMPLE 5000 ROWS
""").df()

print("Kolom akhir:")
print(sample.columns.tolist())

new_cols = [
    "expected_sales_baseline", "baseline_source",
    "hours_into_stockout", "hours_since_available", "stockout_episode_id",
    "sales_lag_7d", "sales_lag_14d", "rolling_mean_7d", "rolling_mean_14d",
]
print("\nNull count per kolom fitur baru (dari sample):")
print(sample[new_cols].isnull().sum())

not_stockout = sample[sample["is_stockout"] == 0]
corr = not_stockout[["expected_sales_baseline", "observed_sales"]].corr().iloc[0, 1]
print(f"\nKorelasi expected_sales_baseline vs observed_sales (saat tidak stockout): {corr:.3f}")
print("(Semakin mendekati 1, semakin baik baseline menangkap pola normal.)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Kolom akhir:
['datetime', 'hour', 'weekday', 'observed_sales', 'is_stockout', 'latent_demand', 'store_id', 'product_id', 'city_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level', 'date', 'expected_sales_baseline', 'baseline_source', 'hours_into_stockout', 'hours_since_available', 'stockout_episode_id', 'sales_lag_7d', 'sales_lag_14d', 'rolling_mean_7d', 'rolling_mean_14d']

Null count per kolom fitur baru (dari sample):
expected_sales_baseline      0
baseline_source              0
hours_into_stockout          0
hours_since_available        0
stockout_episode_id          0
sales_lag_7d               391
sales_lag_14d              799
rolling_mean_7d             71
rolling_mean_14d            71
dtype: int64

Korelasi expected_sales_baseline vs observed_sales (saat tidak stockout): 0.742
(Semakin mendekati 1, semakin baik baseline menangkap p